In [1]:
import math
import time
import numpy as np
import pandas as pd
import yfinance as yf


import torch
import torch.nn as nn

from model import train_pinn


from montecarlo import mc_asian_call_arith
from montecarlo import pinn_price_real


from evaluation import evaluate_mae_pinn_vs_mc
from evaluation import parity_plot_plotly
from evaluation import run_full_evaluation


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32


In [2]:
# Get real WTI data and infer ranges

df = yf.download("CL=F", start="2010-01-01", progress=False)
prices = df["Close"].dropna()


S0 = float(prices.iloc[-1])
S_max_hist = float(prices.max())


log_ret = np.log(prices / prices.shift(1)).dropna()
sigma_hist = float(log_ret.std()) * np.sqrt(252)


T = 1.0
S_max_real = 1.2 * S_max_hist
I_max_real = S_max_real * T
sigma_max = min(1.0, 2.0 * sigma_hist)
r_max = 0.5


print("=== REAL (USD) ranges ===")
print(f"S0        = {S0:.2f}")
print(f"S_max     = {S_max_real:.2f}")
print(f"I_max     = {I_max_real:.2f}")
print(f"sigma     = {sigma_hist:.3f}")
print(f"sigma_max = {sigma_max:.3f}")
print(f"r_max     = {r_max}")


YF.download() has changed argument auto_adjust default to True
=== REAL (USD) ranges ===
S0        = 56.82
S_max     = 148.44
I_max     = 148.44
sigma     = 0.404
sigma_max = 0.807
r_max     = 0.5


/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_78529/968340630.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S0 = float(prices.iloc[-1])
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_78529/968340630.py:8: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  S_max_hist = float(prices.max())
/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_78529/968340630.py:12: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  sigma_hist = float(log_ret.std()) * np.sqrt(252)


In [3]:
# Normalization: K_scale = S0  (so S̃0 = 1)

K_scale = S0  


S_max = S_max_real / K_scale
I_max = I_max_real / K_scale


print("\n=== NORMALIZED (dimensionless) ranges ===")
print(f"K_scale   = {K_scale:.2f}")
print(f"S0_tilde  = {S0 / K_scale:.3f}")      # should be 1.000
print(f"S_max_t   = {S_max:.3f}")
print(f"I_max_t   = {I_max:.3f}")
print(f"r_max     = {r_max:.3f}")
print(f"sigma_max = {sigma_max:.3f}")



=== NORMALIZED (dimensionless) ranges ===
K_scale   = 56.82
S0_tilde  = 1.000
S_max_t   = 2.612
I_max_t   = 2.612
r_max     = 0.500
sigma_max = 0.807


In [4]:
model = train_pinn(
   S_max, I_max, r_max, sigma_max,
   width=160,
   depth=4,
   n_epochs=20000,     # start here; increase toward 200k/500k if needed
   lr0=1e-3,
   Np=2000,
   n_bc_axis=100,
   w_pde=1.0,
   print_every=2000
)

K_real = S0  # ATM
r_eval = 0.05
sigma_eval = sigma_hist



print("Evaluation (real scale)")
test_S = [0.8*S0, 1.0*S0, 1.2*S0]  # around spot


for S_test in test_S:
   pinn_p = pinn_price_real(model, S_test, K_real, r_eval, sigma_eval, t0=0.0)
   mc_p, mc_se = mc_asian_call_arith(S_test, K_real, r_eval, sigma_eval, T=1.0, n_steps=252, n_paths=50_000, antithetic=True)


   print(f"S={S_test:8.2f} | PINN={pinn_p:10.4f} | MC={mc_p:10.4f} ± {1.645*mc_se:8.4f} (90% CI half-width)")


ep=   2000 | loss=6.753e-04 | pde=2.678e-04 | data=4.074e-04 | lr=9.76e-04 | 2.4 min
ep=   4000 | loss=2.337e-04 | pde=1.012e-04 | data=1.324e-04 | lr=9.05e-04 | 4.9 min
ep=   6000 | loss=1.390e-04 | pde=3.744e-05 | data=1.016e-04 | lr=7.94e-04 | 7.5 min
ep=   8000 | loss=7.303e-05 | pde=3.008e-05 | data=4.296e-05 | lr=6.55e-04 | 9.8 min
ep=  10000 | loss=4.387e-05 | pde=2.581e-05 | data=1.807e-05 | lr=5.00e-04 | 12.3 min
ep=  12000 | loss=6.121e-05 | pde=2.597e-05 | data=3.524e-05 | lr=3.45e-04 | 15.4 min
ep=  14000 | loss=2.582e-05 | pde=1.326e-05 | data=1.256e-05 | lr=2.06e-04 | 18.0 min
ep=  16000 | loss=3.251e-05 | pde=1.232e-05 | data=2.018e-05 | lr=9.55e-05 | 20.5 min
ep=  18000 | loss=1.747e-05 | pde=9.102e-06 | data=8.365e-06 | lr=2.45e-05 | 23.1 min
ep=  20000 | loss=1.810e-05 | pde=1.092e-05 | data=7.184e-06 | lr=0.00e+00 | 25.9 min
Evaluation (real scale)
S=   45.46 | PINN=    1.2564 | MC=    1.3528 ±   0.0320 (90% CI half-width)
S=   56.82 | PINN=    5.8408 | MC=    5.8057

In [5]:
S_grid_eval = np.linspace(0, S_max_real, 200)

results = run_full_evaluation(
    model=model,          # PINN già addestrata
    S_grid=S_grid_eval,   # grid su S
    K=K_real,        # strike reale
    r=r_eval,             # risk-free rate
    sigma=sigma_eval,     # volatilità
    T=1.0,                # maturità
    n_steps=252,       # discretizzazione MC
    n_paths=200_000,   # paths MC
    h_delta=1.0,          # bump per delta
)

MAE (PINN vs MC): 0.089016
Speed-up (MC / PINN): 20425.0x
